In [2]:
# imports
import chromadb
from sentence_transformers import SentenceTransformer
from anthropic import Anthropic
from dotenv import load_dotenv
import os

load_dotenv()

True

In [3]:
# Load chromadb
chroma_client = chromadb.PersistentClient("../vectorstore/food_poverty_db/")
collection_foodprice = chroma_client.get_collection(name='food_prices')
collection_poverty = chroma_client.get_collection(name='poverty_mpi')

embed_model = SentenceTransformer('all-MiniLM-L6-v2')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
# building retriever function -that takes a question, searches ChromaDB
# return top 3 relevant documents from each collection and combining them
def retrieve(question, n_results=5):
    question_embedding = embed_model.encode(question).tolist()
    
    food_docs = collection_foodprice.query(
        query_embeddings = [question_embedding],
        n_results = n_results
        )['documents'][0]
    poverty_docs = collection_poverty.query(
        query_embeddings = [question_embedding],
        n_results = n_results
        )['documents'][0]
    
    return food_docs + poverty_docs

In [9]:
# connecting to anthropoic api
anthropic_client = Anthropic()

In [10]:
# building prompt template
def ask(question):
    docs = retrieve(question)
    context_text = '\n'.join(docs)
    prompt = f"""Context: {context_text}
    Question: {question}
    Anwser based only on context provided."""
    
    response = anthropic_client.messages.create(
        model = "claude-haiku-4-5-20251001",
        max_tokens=1024,
        system = """You are a humanitarian data analyst assistant. 
            Answer questions based only on the provided context. 
            Be precise with numbers and facts. 
            Always cite specific figures when available.
            """,
        messages=[{'role': 'user', 'content': prompt}]
    )
    return response.content[0].text
    

In [11]:
# testing woth sample questions
ask('which state has the highest rice prices in 2024?')

'Based on the context provided, I cannot answer this question. \n\nThe context does not contain any data about rice prices in 2024. The only rice price data available is from 2022 and 2023 for Himachal Pradesh. Additionally, there is no rice price information provided for other states (Uttar Pradesh or Arunachal Pradesh) that would allow for a comparison of which state has the highest rice prices.\n\nTo answer your question, I would need rice price data from 2024 for multiple states.'

In [12]:
print(ask("What is the rice price in Assam?"))


# Rice Price in Assam

Based on the context provided, here are the retail prices of rice in Assam:

| Date | Price (INR per KG) |
|------|-------------------|
| 2025-01-15 | 42.63 |
| 2025-06-15 | 42.15 |
| 2025-10-15 | 42.21 |
| 2025-11-15 | 42.2 |
| 2023-10-15 | 42.24 |

**Most recent price (as of 2025-11-15):** **42.2 INR per KG**

The rice prices in Assam have remained relatively stable, ranging between 42.15 to 42.63 INR per KG across the recorded dates.


In [13]:
print(ask("What is the poverty situation and food prices in Bihar?"))

# Poverty Situation and Food Prices in Bihar

## Poverty Situation

Bihar has significant poverty challenges with multiple poverty measures reported:

- **34.66%** of the population lives in poverty with an MPI score of 0.1544
- **52.41%** of the population lives in poverty with an MPI score of 0.2475
- **77.37%** of the population lives in poverty with an MPI score of 0.4484

These varying figures suggest different poverty measurement methodologies or time periods, with the highest measure indicating that over three-quarters of Bihar's population experiences some form of poverty.

## Food Prices (Rice)

The retail price of rice in Bihar showed an increase over the period measured:

- **2004**: 8.0 INR per KG (both in June and September)
- **2008-2009**: 12.0 INR per KG (in October 2008, November 2008, and June 2009)

This represents a **50% increase** in rice prices from 2004 to 2008-2009, which is a significant rise in a staple food commodity in a region with substantial poverty.


In [14]:
print(ask("In bihar, is there any pattern or relation finidngs between poverty and food prices?"))

# Analysis of Poverty and Food Prices in Bihar

Based on the provided context, I cannot establish a clear pattern or causal relationship between poverty and food prices in Bihar.

## Available Data:

**Rice Prices in Bihar:**
- 2003-2004: 8.0 INR per KG (multiple data points)
- 2009: 12.0 INR per KG
- This shows an increase of 4.0 INR per KG over approximately 5-6 years

**Poverty Rates in Bihar:**
- 34.66% of population (MPI: 0.1544)
- 52.41% of population (MPI: 0.2475)
- 77.37% of population (MPI: 0.4484)

## Limitation:

The data provided does not include **temporal alignment** - the poverty statistics lack specific dates, while the rice price data are time-stamped. Without knowing when these poverty measurements were taken, I cannot determine whether poverty levels changed in correlation with rice price changes.

To establish a meaningful relationship, one would need poverty data with specific dates that align with the food price observations.
